In [103]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [104]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [105]:
file_path = '/content/drive/MyDrive/spectra/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv'
df1 = pd.read_csv(file_path)

# Clean column names by stripping whitespace
df1.columns = df1.columns.str.strip()

df1

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,22,1266342,41,44,2664,6954,456,0,64.975610,109.864573,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,22,1319353,41,44,2664,6954,456,0,64.975610,109.864573,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,22,160,1,1,0,0,0,0,0.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,22,1303488,41,42,2728,6634,456,0,66.536585,110.129945,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,35396,77,1,2,0,0,0,0,0.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286462,443,196135,49,57,1331,105841,570,0,27.163265,108.067176,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
286463,443,378424,49,59,1325,104393,570,0,27.040816,108.095051,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
286464,443,161800,70,103,1427,215903,570,0,20.385714,90.746389,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
286465,443,142864,50,62,1331,110185,570,0,26.620000,107.027727,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [106]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 286467 entries, 0 to 286466
Data columns (total 79 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Destination Port             286467 non-null  int64  
 1   Flow Duration                286467 non-null  int64  
 2   Total Fwd Packets            286467 non-null  int64  
 3   Total Backward Packets       286467 non-null  int64  
 4   Total Length of Fwd Packets  286467 non-null  int64  
 5   Total Length of Bwd Packets  286467 non-null  int64  
 6   Fwd Packet Length Max        286467 non-null  int64  
 7   Fwd Packet Length Min        286467 non-null  int64  
 8   Fwd Packet Length Mean       286467 non-null  float64
 9   Fwd Packet Length Std        286467 non-null  float64
 10  Bwd Packet Length Max        286467 non-null  int64  
 11  Bwd Packet Length Min        286467 non-null  int64  
 12  Bwd Packet Length Mean       286467 non-null  float64
 13 

In [107]:
df1['Label'].value_counts()

,count
Label,
PortScan,158930
BENIGN,127537


In [108]:
drop_cols = [
    'Fwd Header Length.1',

    'Fwd Avg Bytes/Bulk',
    'Fwd Avg Packets/Bulk',
    'Fwd Avg Bulk Rate',

    'Bwd Avg Bytes/Bulk',
    'Bwd Avg Packets/Bulk',
    'Bwd Avg Bulk Rate',

    'Fwd URG Flags',
    'Bwd URG Flags',
    'CWE Flag Count',
    'ECE Flag Count',
    'URG Flag Count'
]

df = df1.drop(columns=drop_cols)

In [109]:
X = df.drop(columns=['Label'])

constant_cols = [
    col for col in X.columns
    if X[col].nunique() <= 1
]

print(constant_cols)

['Bwd PSH Flags']


In [110]:
df = df.drop(columns=constant_cols)

In [111]:
X = df.drop(columns=['Label'])

corr = X.corr().abs()

upper = corr.where(
    np.triu(np.ones(corr.shape), k=1).astype(bool)
)

high_corr_cols = [
    column
    for column in upper.columns
    if any(upper[column] > 0.95)
]

print(high_corr_cols)

['Total Backward Packets', 'Total Length of Bwd Packets', 'Flow IAT Std', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length', 'Packet Length Std', 'SYN Flag Count', 'Average Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets', 'Subflow Bwd Bytes', 'Idle Mean', 'Idle Max', 'Idle Min']


In [112]:
drop_corr = [
    'Fwd IAT Total',
    'Fwd IAT Mean',
    'Fwd IAT Max',
    'Fwd IAT Min',

    'Bwd IAT Total',
    'Bwd IAT Mean',
    'Bwd IAT Max',
    'Bwd IAT Min',

    'Fwd Header Length',

    'Subflow Fwd Packets',
    'Subflow Fwd Bytes',
    'Subflow Bwd Packets',
    'Subflow Bwd Bytes',
]
df = df.drop(columns = drop_corr)

In [113]:
df = df.replace([np.inf, -np.inf], np.nan)
df = df.fillna(0)

In [114]:
print(df.shape)
print(df.columns.tolist())

(286467, 53)
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Std', 'Bwd IAT Std', 'Fwd PSH Flags', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'Down/Up Ratio', 'Average Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward', 'Active Mean', 'Active Std', 'Active Max', 'Active 

In [115]:
X = df.drop(columns=['Label'])
y = df['Label'].map({
    'BENIGN': 0,
    'PortScan': 1
})

print(X.shape)
print(y.value_counts())

(286467, 52)
Label
1    158930
0    127537
Name: count, dtype: int64


In [116]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(229173, 52)
(57294, 52)


In [117]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss',
    enable_categorical=True # Ensure categorical features are handled if present
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [118]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [119]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=['BENIGN', 'PortScan']
))

Accuracy: 0.9998952769923553

Confusion Matrix:
[[25507     1]
 [    5 31781]]

Classification Report:
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     25508
    PortScan       1.00      1.00      1.00     31786

    accuracy                           1.00     57294
   macro avg       1.00      1.00      1.00     57294
weighted avg       1.00      1.00      1.00     57294



In [120]:
X.duplicated().sum()

np.int64(72586)

In [121]:
df.groupby('Label').apply(
    lambda x: x.drop(columns=['Label']).duplicated().sum()
)

/tmp/ipykernel_1750/3698684908.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby('Label').apply(


,0
Label,
BENIGN,4475
PortScan,68111


In [122]:
train_rows = set(map(tuple, X_train.values))
test_rows = set(map(tuple, X_test.values))

print("Exact duplicate rows across train/test:",
      len(train_rows.intersection(test_rows)))

Exact duplicate rows across train/test: 15450


In [123]:
duplicate_label_check = (
    df3.groupby(X.columns.tolist())['Label']
       .nunique()
)

print("Feature vectors with conflicting labels:",
      (duplicate_label_check > 1).sum())

Feature vectors with conflicting labels: 0


In [124]:
from sklearn.model_selection import GroupShuffleSplit

groups = X.astype(str).agg('|'.join, axis=1)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print(X_train.shape)
print(X_test.shape)

(229167, 52)
(57300, 52)


In [125]:
train_rows = set(map(tuple, X_train.values))
test_rows = set(map(tuple, X_test.values))

print(
    "Exact duplicate rows across train/test:",
    len(train_rows.intersection(test_rows))
)

Exact duplicate rows across train/test: 0


In [126]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [127]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=['BENIGN', 'PortScan']
))

Accuracy: 0.9999301919720768

Confusion Matrix:
[[25271     0]
 [    4 32025]]

Classification Report:
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     25271
    PortScan       1.00      1.00      1.00     32029

    accuracy                           1.00     57300
   macro avg       1.00      1.00      1.00     57300
weighted avg       1.00      1.00      1.00     57300



In [128]:
importance = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance)

Total Length of Fwd Packets    0.755010
Flow Bytes/s                   0.189985
Min Packet Length              0.018718
Flow Packets/s                 0.008570
Bwd Packet Length Min          0.006495
ACK Flag Count                 0.002896
Bwd Packets/s                  0.002638
Packet Length Std              0.002147
Packet Length Mean             0.001964
Init_Win_bytes_forward         0.001662
Flow IAT Mean                  0.001245
Active Std                     0.000943
Total Fwd Packets              0.000732
Idle Min                       0.000689
Flow IAT Std                   0.000610
Total Length of Bwd Packets    0.000520
Flow Duration                  0.000475
Average Packet Size            0.000371
Fwd Packet Length Max          0.000355
Bwd IAT Std                    0.000316
Fwd IAT Std                    0.000301
Fwd Packet Length Min          0.000296
PSH Flag Count                 0.000268
Bwd Packet Length Mean         0.000249
Destination Port               0.000241


In [129]:
top_features = [
    'Total Length of Fwd Packets',
    'Flow Bytes/s',
    'Min Packet Length',
    'Flow Packets/s',
    'Bwd Packet Length Min',
    'ACK Flag Count',
    'Bwd Packets/s',
    'Packet Length Std',
    'Packet Length Mean',
    'Init_Win_bytes_forward'
]

X_small = df[top_features]

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_small,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [130]:
X_train_s = X_small.iloc[train_idx]
X_test_s = X_small.iloc[test_idx]

y_train_s = y.iloc[train_idx]
y_test_s = y.iloc[test_idx]

In [131]:
model_small = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

model_small.fit(X_train_s, y_train_s)

y_pred_s = model_small.predict(X_test_s)

print(classification_report(
    y_test_s,
    y_pred_s,
    target_names=['BENIGN', 'PortScan']
))

              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     25271
    PortScan       1.00      1.00      1.00     32029

    accuracy                           1.00     57300
   macro avg       1.00      1.00      1.00     57300
weighted avg       1.00      1.00      1.00     57300



In [132]:
print("Accuracy:", accuracy_score(y_test_s, y_pred_s))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_s, y_pred_s))

print("\nClassification Report:")
print(classification_report(
    y_test_s,
    y_pred_s,
    target_names=['BENIGN', 'PortScan'],
    digits=6
))


Accuracy: 0.999912739965096

Confusion Matrix:
[[25270     1]
 [    4 32025]]

Classification Report:
              precision    recall  f1-score   support

      BENIGN   0.999842  0.999960  0.999901     25271
    PortScan   0.999969  0.999875  0.999922     32029

    accuracy                       0.999913     57300
   macro avg   0.999905  0.999918  0.999912     57300
weighted avg   0.999913  0.999913  0.999913     57300



In [133]:
top_features_no_dominant = [
    'Min Packet Length',
    'Flow Packets/s',
    'Bwd Packet Length Min',
    'ACK Flag Count',
    'Bwd Packets/s',
    'Packet Length Std',
    'Packet Length Mean',
    'Init_Win_bytes_forward'
]
X_small2 = df[top_features_no_dominant]

X_train_s2 = X_small2.iloc[train_idx]
X_test_s2 = X_small2.iloc[test_idx]

y_train_s2 = y.iloc[train_idx]
y_test_s2 = y.iloc[test_idx]

model_small2 = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

model_small2.fit(X_train_s2, y_train_s2)

y_pred_s2 = model_small2.predict(X_test_s2)

print("Accuracy:", accuracy_score(y_test_s2, y_pred_s2))

print(confusion_matrix(y_test_s2, y_pred_s2))

print(classification_report(
    y_test_s2,
    y_pred_s2,
    target_names=['BENIGN', 'PortScan'],
    digits=6
))

Accuracy: 0.9997905759162303
[[25265     6]
 [    6 32023]]
              precision    recall  f1-score   support

      BENIGN   0.999763  0.999763  0.999763     25271
    PortScan   0.999813  0.999813  0.999813     32029

    accuracy                       0.999791     57300
   macro avg   0.999788  0.999788  0.999788     57300
weighted avg   0.999791  0.999791  0.999791     57300



In [135]:
portscan_features = [
    'Destination Port',
    'Flow Duration',
    'Total Fwd Packets',
    'Total Backward Packets',
    'Flow Packets/s',
    'Fwd Packets/s',
    'Bwd Packets/s',
    'SYN Flag Count',
    'RST Flag Count',
    'ACK Flag Count',
    'FIN Flag Count'
]
X_ps = df[portscan_features]

X_train_ps = X_ps.iloc[train_idx]
X_test_ps = X_ps.iloc[test_idx]

y_train_ps = y.iloc[train_idx]
y_test_ps = y.iloc[test_idx]

model_ps = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

model_ps.fit(X_train_ps, y_train_ps)

y_pred_ps = model_ps.predict(X_test_ps)

print("Accuracy:", accuracy_score(y_test_ps, y_pred_ps))

print(confusion_matrix(y_test_ps, y_pred_ps))

print(classification_report(
    y_test_ps,
    y_pred_ps,
    target_names=['BENIGN', 'PortScan'],
    digits=6
))

Accuracy: 0.9998429319371728
[[25267     4]
 [    5 32024]]
              precision    recall  f1-score   support

      BENIGN   0.999802  0.999842  0.999822     25271
    PortScan   0.999875  0.999844  0.999860     32029

    accuracy                       0.999843     57300
   macro avg   0.999839  0.999843  0.999841     57300
weighted avg   0.999843  0.999843  0.999843     57300



In [136]:
importance_ps = pd.Series(
    model_ps.feature_importances_,
    index=X_train_ps.columns
).sort_values(ascending=False)

print(importance_ps)

Flow Duration             0.518097
ACK Flag Count            0.426528
Total Fwd Packets         0.028761
Total Backward Packets    0.008251
FIN Flag Count            0.005795
Destination Port          0.005690
Bwd Packets/s             0.004741
Fwd Packets/s             0.001213
Flow Packets/s            0.000923
RST Flag Count            0.000000
SYN Flag Count            0.000000
dtype: float32


In [137]:
print(
    df.groupby('Label')['ACK Flag Count']
       .describe()
)

             count      mean       std  min  25%  50%  75%  max
Label                                                          
BENIGN    127537.0  0.278304  0.448165  0.0  0.0  0.0  1.0  1.0
PortScan  158930.0  0.000384  0.019588  0.0  0.0  0.0  0.0  1.0


In [138]:
print(
    df.groupby('Label')['Flow Duration']
       .describe()
)

             count          mean           std   min    25%      50%  \
Label                                                                  
BENIGN    127537.0  1.197957e+07  3.153322e+07 -13.0  186.0  30964.0   
PortScan  158930.0  8.282023e+04  2.325855e+06   0.0   42.0     47.0   

               75%          max  
Label                            
BENIGN    551644.0  119999949.0  
PortScan      61.0  119809735.0  


In [139]:
print(
    df.groupby('Label')['SYN Flag Count']
       .value_counts()
)

Label     SYN Flag Count
BENIGN    0                 121523
          1                   6014
PortScan  0                 158930
Name: count, dtype: int64


In [140]:
print(
    df.groupby('Label')['RST Flag Count']
       .value_counts()
)

Label     RST Flag Count
BENIGN    0                 127516
          1                     21
PortScan  0                 158930
Name: count, dtype: int64


In [141]:
portscan_features_2 = [
    'Destination Port',
    'Flow Duration',
    'Total Fwd Packets',
    'Total Backward Packets',
    'Flow Packets/s',
    'Fwd Packets/s',
    'Bwd Packets/s',
    'FIN Flag Count'
]

X_ps2 = df[portscan_features_2]

X_train_ps2 = X_ps2.iloc[train_idx]
X_test_ps2 = X_ps2.iloc[test_idx]

y_train_ps2 = y.iloc[train_idx]
y_test_ps2 = y.iloc[test_idx]

model_ps2 = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

model_ps2.fit(X_train_ps2, y_train_ps2)

y_pred_ps2 = model_ps2.predict(X_test_ps2)

print("Accuracy:", accuracy_score(y_test_ps2, y_pred_ps2))
print(confusion_matrix(y_test_ps2, y_pred_ps2))

print(classification_report(
    y_test_ps2,
    y_pred_ps2,
    target_names=['BENIGN', 'PortScan'],
    digits=6
))

Accuracy: 0.9808027923211169
[[24534   737]
 [  363 31666]]
              precision    recall  f1-score   support

      BENIGN   0.985420  0.970836  0.978074     25271
    PortScan   0.977255  0.988667  0.982928     32029

    accuracy                       0.980803     57300
   macro avg   0.981338  0.979751  0.980501     57300
weighted avg   0.980856  0.980803  0.980787     57300



In [142]:
importance_ps2 = pd.Series(
    model_ps2.feature_importances_,
    index=X_train_ps2.columns
).sort_values(ascending=False)

print(importance_ps2)

Flow Duration             0.515203
Total Fwd Packets         0.269789
Total Backward Packets    0.068339
FIN Flag Count            0.049080
Bwd Packets/s             0.044744
Destination Port          0.035255
Fwd Packets/s             0.010036
Flow Packets/s            0.007554
dtype: float32


In [143]:
portscan_features_3 = [
    'Flow Duration',
    'Total Fwd Packets',
    'Total Backward Packets',
    'FIN Flag Count',
    'Destination Port'
]

X_ps3 = df[portscan_features_3]

X_train_ps3 = X_ps3.iloc[train_idx]
X_test_ps3 = X_ps3.iloc[test_idx]

y_train_ps3 = y.iloc[train_idx]
y_test_ps3 = y.iloc[test_idx]

model_ps3 = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

model_ps3.fit(X_train_ps3, y_train_ps3)

y_pred_ps3 = model_ps3.predict(X_test_ps3)

print("Accuracy:", accuracy_score(y_test_ps3, y_pred_ps3))

print(confusion_matrix(y_test_ps3, y_pred_ps3))

print(classification_report(
    y_test_ps3,
    y_pred_ps3,
    target_names=['BENIGN', 'PortScan'],
    digits=6
))

Accuracy: 0.9807155322862129
[[24537   734]
 [  371 31658]]
              precision    recall  f1-score   support

      BENIGN   0.985105  0.970955  0.977979     25271
    PortScan   0.977340  0.988417  0.982847     32029

    accuracy                       0.980716     57300
   macro avg   0.981223  0.979686  0.980413     57300
weighted avg   0.980765  0.980716  0.980700     57300



In [144]:
import joblib

joblib.dump(model_ps3, "portscan_model.pkl")

print("Saved successfully:", "portscan_model.pkl")

Saved successfully: portscan_model.pkl
